# DPO (Direct Preference Optimization) 학습

- **목적**: SFT 모델을 기반으로 `(prompt, chosen, rejected)` 3쌍 데이터를 활용해 선호도 학습
- **베이스 모델**: `Gyu96/llama3.1_additional_checkpoint1100` (SFT 최종 체크포인트)
- **학습 방식**: QLoRA (4bit 양자화 + LoRA 어댑터)
- **입력 데이터**: `dpo_preprocessing.ipynb`에서 구성한 `DPO_dataset_final.csv`

## DPO 학습 구조

| 컬럼 | 설명 |
|------|------|
| `prompt` | 난독화 문장 + 시스템 프롬프트 |
| `chosen` | 원본 문장 (정답) |
| `rejected` | SFT 모델이 틀리게 예측한 문장 (오답) |

- `ref_model`: SFT 최종 체크포인트를 참조 모델로 명시적 지정 → DPO loss 계산 기준점
- `beta=0.1`: 참조 모델로부터 벗어나는 정도 조절 (0에 가까울수록 참조 모델 영향 감소)

In [ ]:
!pip install accelerate==1.2.1 bitsandbytes==0.45.1 peft==0.14.0 transformers==4.48.2 trl==0.14.0
!pip install flash-attn --no-build-isolation
!pip install vllm==0.7.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd

# ── 경로 설정 ─────────────────────────────────────────────────────────────
BASE_DIR   = '/content/drive/MyDrive/Colab Notebooks/open'
CKPT_DIR   = '/content/drive/MyDrive/Colab Notebooks/model_checkpoint/DPO_training'
INPUT_CSV  = os.path.join(BASE_DIR, 'DPO_dataset_final.csv')

os.makedirs(CKPT_DIR, exist_ok=True)

## 1. 데이터 로드

In [ ]:
dpo_data = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
print(f'DPO 데이터 크기: {len(dpo_data)}')
dpo_data.head(3)

## 2. 모델 로드 (QLoRA)

- SFT 최종 체크포인트를 베이스로 DPO 학습 진행
- SFT 대비 LoRA rank를 r=32로 높여 선호도 학습에서 더 넓은 표현력 확보
- Flash Attention 2로 긴 시퀀스 처리 효율화

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
import torch

MODEL_NAME = 'Gyu96/llama3.1_additional_checkpoint1100'

# 4bit 양자화 설정
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

# LoRA 설정 (SFT r=16 → DPO r=32로 확장)
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    attn_implementation='flash_attention_2',
    device_map={'': 0}
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({'pad_token': '<|reserved_special_token_247|>'})
model.config.pad_token_id = tokenizer.pad_token_id

## 3. 프롬프트 포맷 설정

- SFT와 동일한 난독화 복원 전용 포맷 사용
- DPO는 `(prompt, chosen, rejected)` 3쌍 구조로 데이터셋 구성

In [ ]:
EOS_TOKEN = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
    "Your task is to transform the given obfuscated Korean review into a clear, correct, "
    "and natural-sounding Korean review that reflects its original meaning. "
    "Spacing and word length in the output must be restored to the same as in the input. "
    "Do not provide any description. Print only in Korean."
)

def build_deobfuscation_prompt(instruction, response):
    return (
        f"{SYSTEM_PROMPT}\n"
        f"\n\n### Instruction:\n{instruction}"
        f"\n\n### Response:\n{response}    "
    )

def formatting_prompts_func_for_dpo(examples):
    prompt_lst, chosen_lst, rejected_lst = [], [], []

    for prompt, chosen, rejected in zip(examples['input'], examples['output'], examples['inference']):
        prompt_lst.append(build_deobfuscation_prompt(prompt, ''))
        chosen_lst.append(chosen + EOS_TOKEN)
        rejected_lst.append(rejected + EOS_TOKEN)

    return {'prompt': prompt_lst, 'chosen': chosen_lst, 'rejected': rejected_lst}

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(dpo_data)
dataset = dataset.shuffle(seed=42)

mapped_dataset = dataset.map(formatting_prompts_func_for_dpo, batched=True)
mapped_dataset = mapped_dataset.select_columns(['prompt', 'chosen', 'rejected'])
split_dataset  = mapped_dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = split_dataset['train']
test_dataset  = split_dataset['test']

print(f'Train: {len(train_dataset)} | Test: {len(test_dataset)}')

# 데이터 구조 확인
print('\n[prompt]:', train_dataset[0]['prompt'])
print('[chosen]:', train_dataset[0]['chosen'])
print('[rejected]:', train_dataset[0]['rejected'])

## 4. 학습 설정 및 실행

- `DPOTrainer` 사용 (trl 라이브러리)
- `ref_model`: SFT 최종 체크포인트를 참조 모델로 명시적 지정 → DPO loss 계산 기준점
- `beta=0.1`: 참조 모델 영향도 조절 (낮을수록 참조 모델 무시)
- SFT 대비 학습률을 `1e-4 → 5e-5`로 낮춰 안정적인 선호도 학습

In [ ]:
%load_ext tensorboard
%tensorboard --logdir '{CKPT_DIR}/runs'

In [ ]:
# SFT 체크포인트를 참조 모델로 로드 (DPO loss 계산 기준점)
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={'': 0}
)

from trl import DPOTrainer, DPOConfig

training_arguments = DPOConfig(
    output_dir=CKPT_DIR,
    report_to='tensorboard',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,  # 유효 배치 = 2 * 16 = 32
    warmup_steps=50,
    max_steps=500,
    eval_steps=50,
    save_steps=50,
    evaluation_strategy='steps',
    save_strategy='steps',
    learning_rate=5e-5,              # SFT(1e-4)보다 낮게 설정
    logging_steps=1,
    optim='adamw_8bit',
    weight_decay=0.01,
    max_length=4096,
    lr_scheduler_type='constant_with_warmup',
    seed=42,
    dataset_num_proc=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
    beta=0.1
)

trainer = DPOTrainer(
    model,
    tokenizer=tokenizer,
    ref_model=ref_model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments
)

In [ ]:
trainer.train()

## 5. 모델 병합 및 HuggingFace 업로드

- QLoRA 학습 후 LoRA 어댑터를 base 모델에 병합 (`merge_and_unload`)
- 병합된 full 모델을 HuggingFace에 업로드 → vLLM 추론 가능 상태
- validation loss 기준 최적 체크포인트인 `checkpoint-200` 선택

In [ ]:
from peft import PeftModel

MERGED_MODEL_ID = 'Gyu96/llama3.1_DPO_checkpoint200'
FINAL_CKPT_PATH = os.path.join(CKPT_DIR, 'checkpoint-200')

# 1. SFT merged 모델을 bfloat16으로 재로드 (DPO LoRA merge 용도)
#    DPO는 SFT 체크포인트 위에 학습했으므로 base는 SFT merged 모델
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={'': 0}
)

# 2. DPO LoRA 어댑터 병합 후 어댑터 제거
base_model   = PeftModel.from_pretrained(base_model, FINAL_CKPT_PATH)
merged_model = base_model.merge_and_unload()

# 3. 병합된 full 모델 업로드
merged_model.push_to_hub(MERGED_MODEL_ID, token=True)
tokenizer.push_to_hub(MERGED_MODEL_ID, token=True)

## 6. DPO 모델 추론

- DPO 체크포인트(`llama3.1_DPO_checkpoint200`)를 vLLM으로 로드해 테스트 데이터 추론
- `max_tokens`는 입력 문장 길이 기준으로 동적 설정 → 불필요한 생성 방지
- 추론 결과를 `DPO_inference_results.csv`로 저장

In [ ]:
import pandas as pd
from tqdm import tqdm
from vllm import LLM, SamplingParams

DPO_INFER_MODEL = 'Gyu96/llama3.1_DPO_checkpoint200'
DPO_OUTPUT_CSV  = os.path.join(BASE_DIR, 'DPO_inference_results.csv')

DPO_PROMPT = (
    "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
    "Your task is to transform the given obfuscated Korean review into a clear, correct, "
    "and natural-sounding Korean review that reflects its original meaning. "
    "Spacing and word length in the output must be restored to the same as in the input. "
    "Do not provide any description. Print only in Korean."
    "\n\n### Instruction:\n{}\n\n### Response:\n{}"
)

vllm_dpo = LLM(
    model=DPO_INFER_MODEL,
    tokenizer=DPO_INFER_MODEL,
    gpu_memory_utilization=0.9,
    max_model_len=8192
)

In [ ]:
def batch_inference(df, vllm_model, prompt_template, batch_size=128):
    """배치 단위로 추론을 실행하고 결과 리스트를 반환한다.
    입력 문장 길이 기준으로 max_tokens를 동적 설정해 불필요한 생성을 방지.
    """
    results = []

    for i in tqdm(range(0, len(df), batch_size), desc='Inference'):
        batch = df['input'].iloc[i:i + batch_size].tolist()

        max_tokens = max(len(text) for text in batch)
        sampling_params = SamplingParams(
            temperature=0.05,
            top_p=0.95,
            max_tokens=max_tokens
        )

        prompts = [prompt_template.format(text, '') for text in batch]
        outputs = vllm_model.generate(prompts, sampling_params)
        results.extend([o.outputs[0].text.strip() for o in outputs])

    return results

In [ ]:
# test_dataset → pandas DataFrame 변환
test_df = split_dataset['test'].to_pandas()

dpo_preds = batch_inference(test_df, vllm_dpo, DPO_PROMPT)

test_df['inference'] = dpo_preds
test_df.to_csv(DPO_OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'DPO 추론 완료 → {DPO_OUTPUT_CSV}')
test_df[['input', 'output', 'inference']].head(5)